# Diagrama P-v modelado con PRSV para Deueterio (D₂)
---

In [2]:
! pip install thermo
! pip install chemicals

   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ------------- -------------------------- 2.1/6.4 MB 11.7 MB/s eta 0:00:01
   --------------------------- ------------ 4.5/6.4 MB 11.7 MB/s eta 0:00:01
   -------------------------------- ------- 5.2/6.4 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 6.4/6.4 MB 8.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import thermo
import chemicals
from scipy.integrate import quad

CAS_NUMBER = chemicals.search_chemical('deuterium').CASs
Pc =  chemicals.Pc(CAS_NUMBER) # [Pa]
Tc = chemicals.Tc(CAS_NUMBER) # [K]
k1 = 0 #fuente que estudio el uso de diferentes ecuaciones de estado para el hidrogeno, asigno esto, y el hidrogeno similar al deuterio
w = chemicals.omega(CAS_NUMBER)
R = 8.3145 # [J/molK]



18.724 17189.0972184


* PSRV for P


In [103]:
def alfaF(T, Tc, w, k1):
    m = 0.378893 + 1.4897153*w -0.171384*w**2 + 0.0196554*w**3
    return (1+m*(1-(T/Tc)**0.5) + k1*(1-(T/Tc))*(0.7-(T/Tc)))**2

def PRSV_P(v, alfa, T, R, Tc, Pc):
    ac = 0.45724*((R*Tc**2)/Pc)
    b = 0.07780*R*Tc/Pc
    return (R*T/(v-b)) + ((ac*alfa)/(v*(v+b) + b*(v-b)))

def PRSV_V(alfa, T, R, Tc, P, Pc):
    ac = 0.45724*(R*Tc**2/Pc)
    b = 0.07780*R*Tc/Pc
    a = ac*alfa
    C3 = 1.0
    C2 = b - (R * T) / P
    C1 = (a / P) - 3 * (b**2) - (2 * b * R * T) / P
    C0 = (b**3) + (b**2 * R * T) / P - (a * b) / P    
    roots = np.roots([C3, C2, C1, C0])
    
    real_roots = np.real(roots[np.isclose(np.imag(roots), 0, atol=1e-10)])
    real_roots = real_roots[real_roots > b]
    return real_roots

# alfa = alfaF(18.724, Tc, w, k1)
# print(PSRV_V(alfa, 18.724, R, Tc, 17189.0972184, Pc))

 



* Criterio de Maxwell para el Equilibrio


In [111]:
def maxwellAreas(vv, vl, P, T, R, Tc, Pc, alfa):
    return P*(vv - vl) - quad(PRSV_P,  vl, vv, args=(alfa, T, R, Tc, Pc))[0]

* Calculo de fugacidad para PRSV
 

In [112]:
def func(v,T, alfa, ac, b, R):
    return v*((-R*T)/(v-b)**2 - (ac*alfa*(2*v + 2*b)/((v**2+v*b+b*v-b**2)**2)))


def f(alfa, T, R, v):
    vinf = np.max(PRSV_V(alfa, T, R, Tc, 5, Pc))
    ac = 0.45724*(8.3145*Tc**2.5/Pc)
    b = 0.07780*R*Tc/Pc
    return np.exp((quad(func, vinf, v, args=(T, alfa, ac, b, R))/(R*T))[0]) * (Pc/100)


* PRSV for V and Execution

In [104]:
from scipy.optimize import brentq

T = 20 #K

alfa = alfaF(T, Tc, w, k1)

def maxwellAreas(P, T, R, Tc, Pc, alfa):
    roots = PRSV_V(alfa, T, R, Tc, P, Pc)
    if len(roots) < 3:
        return 1.0 
    return P*(np.max(roots) - np.min(roots)) - quad(PRSV_P,  np.min(roots), np.max(roots), args=(alfa, T, R, Tc, Pc))[0]

try:
    # Find the exact P where fugacity difference is zero
    Psat = brentq(maxwellAreas, 0.001, Pc, args=(T, R, Tc, Pc, alfa))
    # Now you have the EXACT pressure to draw your horizontal line!
    roots_at_sat = PRSV_V(alfa, T, R, Tc, Psat, Pc)
    v_liquid = np.min(roots_at_sat)
    v_vapor = np.max(roots_at_sat)
    
    # Plot the horizontal tie-line
    plt.plot([v_liquid, v_vapor], [Psat/1e6, Psat/1e6], 'k--') 
except Exception as e: 
    print(e)
    pass # Temperature might be above critical or search range



f(a) and f(b) must have different signs


In [ ]:
ac = 0.457235 * (R**2)* (Tc**2) / Pc # como en el articulo de PRSV
b = 0.077796 * R* Tc / Pc

temperatures = np.linspace(5, 48, 11) #[K]

pressures = np.linspace(0.05e6, 3.5e6, 100000) #[Pa]

plt.figure(figsize=(10, 8))

for T in temperatures:
    Tr = T / Tc
    alfa = alfaF(T, Tc, w, k1)
    a = ac * alfa
    
    v_plot = []
    p_plot = []
    
    for P in pressures:
        C3 = 1.0
        C2 = b - (R * T) / P
        C1 = (a / P) - 3 * (b**2) - (2 * b * R * T) / P
        C0 = (b**3) + (b**2 * R * T) / P - (a * b) / P
        
        roots = np.roots([C3, C2, C1, C0])
        
        real_roots = np.real(roots[np.isclose(np.imag(roots), 0, atol=1e-10)])
        real_roots = real_roots[real_roots > b]

        if len(real_roots) > 1:
            if maxwellAreas(np.max(real_roots),np.min(real_roots), P, T,R, Tc, Pc, alfa):
                print('.')
                v_plot.append(np.max(real_roots))
                v_plot.append(np.min(real_roots))
                p_plot.append(P / 1e6) 
                p_plot.append(P / 1e6) 
            else:
                print('*')
                p_plot.append(P / 1e6) 
                fv = f(alfa, T, R, np.max(real_roots))
                fl = f(alfa, T, R, np.min(real_roots))
                if fv < fl:
                    v_plot.append(np.max(real_roots))
                elif fl < fv:
                    v_plot.append(np.min(real_roots)) 
        else: 
            v_plot.append(real_roots[0])
            p_plot.append(P / 1e6) 

    v_plot = np.array(v_plot)
    p_plot = np.array(p_plot)
    
    sort_indices = np.argsort(v_plot)
    v_plot_sorted = v_plot[sort_indices]
    p_plot_sorted = p_plot[sort_indices]
    
    if np.isclose(T, Tc, atol=1.0):
        plt.plot(v_plot_sorted, p_plot_sorted, label=f'T = {T:.1f} K (Near Critical)', linewidth=2.5, color='black')
    else:
        plt.plot(v_plot_sorted, p_plot_sorted, label=f'T = {T:.1f} K')

plt.title('P-v Diagram para Deuterio con PRSV', fontsize=14)
plt.xlabel('v (m^3/mol)', fontsize=12)
plt.ylabel('P (MPa)', fontsize=12)

plt.ylim(0, 3.5)
plt.xlim(b, b * 100)    
plt.xscale('log')

plt.axhline(Pc / 1e6, color='gray', linestyle='--', alpha=0.7, label='Pc')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()

plt.show()


.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
